# Bloom Agent — HairDeck (cabelo por IA, LOCAL no Colab)

FOTO → cartas de cabelo estilizado **prontas pra animar** (spring bones) → GLB + VRM
com a **COR extraída da foto**. Paper arXiv 2610.40599, repo hyz317/HairDeck.

**Pré-requisito ÚNICO (uma vez):** conta grátis no Hugging Face →
1. aceite a licença de [facebook/sam3](https://huggingface.co/facebook/sam3) e
   [facebook/dinov3-vitl16-pretrain-lvd1689m](https://huggingface.co/facebook/dinov3-vitl16-pretrain-lvd1689m);
2. crie um token de LEITURA em hf.co/settings/tokens.

**Uso:** no Blender, peça o cabelo por IA (o agent cria o job e mostra `job_id` e `base_url`),
cole os dois na célula 1 e execute as células na ordem. No fim, de volta ao Blender: **Coletar**.
GPU: use o runtime **T4 GPU** (grátis). A instalação demora ~5-10 min (só na 1ª vez).


In [ ]:
# Celula 1 — conecte no job da ponte
job_id = ""    # ex.: 20261108-170000-a1b2c3
base_url = ""  # ex.: https://xxxx.ngrok-free.app

import json, urllib.request
base_url = base_url.rstrip("/")
def _get(url):
    with urllib.request.urlopen(url, timeout=60) as r: return r.read()
def _post(url, data, headers=None):
    req = urllib.request.Request(url, data=data, method="POST", headers=headers or {})
    with urllib.request.urlopen(req, timeout=300) as r: return json.loads(r.read().decode("utf-8"))
print("health:", json.loads(_get(base_url + "/health").decode("utf-8")))
job = json.loads(_get(f"{base_url}/job/{job_id}").decode("utf-8"))
print("job:", job.get("kind"), "| params:", job.get("params"))


In [ ]:
# Celula 2 — ambiente (GPU + HairDeck + dependencias)
import subprocess, sys
from pathlib import Path
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout or "SEM GPU — use o runtime T4!")
if not Path("/content/HairDeck").exists():
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/hyz317/HairDeck.git", "/content/HairDeck"], check=True)
%cd /content/HairDeck
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "pyrender==0.1.45"], check=True)
subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg"], check=True)
print("ambiente pronto ✅")


In [ ]:
# Celula 3 — pesos (precisa do token HF; SAM3+DINOv3 sao GATED — aceite as licencas!)
import getpass, os, subprocess, sys
token = getpass.getpass("Token do Hugging Face (hf_...): ")
os.environ["HF_TOKEN"] = token
subprocess.run([sys.executable, "-c",
    "from huggingface_hub import login; login(token=__import__('os').environ['HF_TOKEN'])"], check=True)
os.environ.setdefault("HAIRDECK_ROOT", "/content/hairdeck_root")
subprocess.run([sys.executable, "scripts/setup/download_weights.py"], check=True)
print("pesos prontos ✅")


In [ ]:
# Celula 4 — envie a FOTO do personagem (a mesma do painel do Bloom)
from google.colab import files
up = files.upload()  # escolha o PNG/JPG do personagem
img_path = list(up.keys())[0]
print("imagem:", img_path)


In [ ]:
# Celula 5 — GERA o cabelo (4 variacoes; ~1-3 min na T4)
N = 4
res = f"/content/hairdeck_out/{Path(img_path).stem}"
subprocess.run([sys.executable, "scripts/image_to_hair.py", img_path,
                "--out", res, "--n", str(N)], check=True)
import os
for raiz, _, arqs in os.walk(res):
    for a in arqs:
        p = os.path.join(raiz, a)
        print(f"{p}  ({os.path.getsize(p)//1024} KB)")


In [ ]:
# Celula 6 — (opcional) VRM da variacao escolhida, com spring bones + cores
VARIACAO = 0   # veja os previews da celula anterior e escolha 0..N-1
subprocess.run([sys.executable, "-m", "hairdeck.vrm.export", res, str(VARIACAO)], check=True)
print("VRM exportado ✅")


In [ ]:
# Celula 7 — devolve GLB (+VRM) pra ponte e finaliza o job
import os, json
results = {}
for raiz, _, arqs in os.walk("/content/hairdeck_out"):
    for a in arqs:
        if a.lower().endswith((".glb", ".vrm")):
            p = os.path.join(raiz, a)
            results[a] = open(p, "rb").read()
assert results, "nenhum GLB/VRM gerado — rode as celulas 5-6"
sent = []
for nome, data in results.items():
    resp = _post(f"{base_url}/job/{job_id}/result", data, {"X-Result-Name": nome})
    print(nome, "->", resp)
    sent.append(nome)
complete = _post(f"{base_url}/job/{job_id}/complete",
                 json.dumps({"status": "ok", "files": sent}).encode("utf-8"),
                 {"Content-Type": "application/json"})
print("complete:", complete)
print("\n✅ PRONTO — volte ao Blender e clique em Coletar com o job_id", job_id,
      "\nO Bloom importa o GLB, posiciona na cabeça e marca como peça cabelo sozinho.")
